# Feature Engineering Combination

Jalankan setelah tiga notebook Extraction selesai. Notebook ini menggabungkan tujuh kelompok fitur pada 50.000 training resmi; official test tidak digunakan. HOG, LBP, HSV, dan statistik RGB dihitung dengan NumPy/matplotlib yang sudah tersedia.

## 1. Konfigurasi
Gunakan run ID identik dengan notebook RGB, AVG, NTSC.

In [1]:
from pathlib import Path
import hashlib
import time

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RUN_ID = "cifar10_7fitur_v1"
SEED = 30092026
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / "fusion"

for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

I0000 00:00:1791489308.840479  208099 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791489308.878521  208099 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791489309.791485  208099 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 2. Data dan split
Gunakan bagian training resmi; cocokkan indeks dan label dari ketiga archive.

In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
assert x_all.shape == (50000, 32, 32, 3)

train_idx, val_idx = train_test_split(
    np.arange(len(y_all)),
    test_size=0.2,
    stratify=y_all,
    random_state=SEED,
)
train_idx = np.sort(train_idx)
val_idx = np.sort(val_idx)
assert len(train_idx) == 40000
assert len(val_idx) == 10000

split_hash = hashlib.sha256(
    train_idx.tobytes() + val_idx.tobytes()
).hexdigest()
print("Split:", len(train_idx), len(val_idx), split_hash[:16])

Split: 40000 10000 16004a1ac8a89ac9


## 3. Arsip CNN
Periksa hash CNN sumber, urutan sampel, label, split, dan versi preprocessing.

In [3]:
cnn_features = {}
model_hashes = {}
for variant in ("rgb", "avg", "ntsc"):
    source = ROOT / "outputs" / RUN_ID / variant
    archive_path = source / "fitur" / "train_features.npz"
    model_path = source / "model" / "cnn.keras"
    assert archive_path.exists() and model_path.exists(), f"Artefak {variant} belum ada"
    with np.load(archive_path, allow_pickle=False) as archive:
        assert str(archive["run_id"]) == RUN_ID and str(archive["variant"]) == variant
        assert str(archive["split_hash"]) == split_hash
        assert str(archive["preprocessing_version"]) == "v1"
        assert np.array_equal(archive["indices"], np.arange(50000))
        assert np.array_equal(archive["y"], y_all)
        assert np.array_equal(archive["train_idx"], train_idx)
        assert np.array_equal(archive["val_idx"], val_idx)
        features = archive["features"].astype(np.float32)
        expected_hash = str(archive["model_sha"])
    actual_hash = hashlib.sha256(model_path.read_bytes()).hexdigest()
    assert actual_hash == expected_hash and features.shape == (50000, 512)
    assert np.isfinite(features).all()
    cnn_features[variant] = features
    model_hashes[variant] = actual_hash
print("Tiga model dan arsip cocok")

Tiga model dan arsip cocok


## 4. Definisi descriptor
HOG 324 dimensi; LBP 256; HSV 48; statistik RGB 9. Gambar konstan menghasilkan vektor berhingga.

In [4]:
def handcrafted_descriptors(x, batch_size=128, verbose=1):
    # HOG: 9 orientasi unsigned, cell 8x8, block 2x2, L2 per block.
    # LBP: 8 tetangga radius 1, histogram 256 kode.
    # HSV: 16 bin per kanal. Statistik RGB: mean, std, skewness per kanal.
    import matplotlib.colors as mcolors
    outputs = {k: [] for k in ("hog", "lbp", "hsv", "rgb_stats")}
    total_batches = int(np.ceil(len(x) / batch_size))
    progress_interval = max(1, total_batches // 20)
    for begin in range(0, len(x), batch_size):
        rgb = np.asarray(x[begin:begin+batch_size], dtype=np.float32) / 255.0
        gray = np.sum(rgb * np.array([0.299, 0.587, 0.114], np.float32), axis=-1)
        gy, gx = np.gradient(gray, axis=(1, 2))
        magnitude = np.sqrt(gx*gx + gy*gy)
        orientation = np.mod(np.arctan2(gy, gx) * (180.0 / np.pi), 180.0)
        bins = np.minimum((orientation / 20.0).astype(np.int32), 8)
        cell_hist = np.zeros((len(rgb), 4, 4, 9), dtype=np.float32)
        for row in range(4):
            for col in range(4):
                b = bins[:, row*8:(row+1)*8, col*8:(col+1)*8]
                m = magnitude[:, row*8:(row+1)*8, col*8:(col+1)*8]
                for angle in range(9):
                    cell_hist[:, row, col, angle] = np.sum(m * (b == angle), axis=(1, 2))
        blocks = []
        for row in range(3):
            for col in range(3):
                block = cell_hist[:, row:row+2, col:col+2, :].reshape(len(rgb), -1)
                denominator = np.maximum(
                    np.linalg.norm(block, axis=1, keepdims=True), 1e-8
                )
                blocks.append(block / denominator)
        outputs["hog"].append(np.concatenate(blocks, axis=1))
        padded = np.pad(gray, ((0, 0), (1, 1), (1, 1)), mode="edge")
        lbp = np.zeros(gray.shape, dtype=np.uint8)
        offsets = [(-1,-1), (-1,0), (-1,1), (0,1), (1,1), (1,0), (1,-1), (0,-1)]
        for bit, (dy, dx) in enumerate(offsets):
            neighbor = padded[:, 1+dy:33+dy, 1+dx:33+dx]
            lbp |= (neighbor >= gray).astype(np.uint8) << bit
        hist = np.zeros((len(rgb), 256), dtype=np.float32)
        np.add.at(hist, (np.repeat(np.arange(len(rgb)), 1024), lbp.reshape(-1)), 1)
        outputs["lbp"].append(hist / 1024.0)
        hsv = mcolors.rgb_to_hsv(rgb)
        hsv_hist = np.zeros((len(rgb), 3, 16), dtype=np.float32)
        for channel in range(3):
            hsv_bin = np.minimum((hsv[..., channel] * 16).astype(np.int32), 15)
            np.add.at(hsv_hist[:, channel, :],
                      (np.repeat(np.arange(len(rgb)), 1024), hsv_bin.reshape(-1)), 1)
        outputs["hsv"].append(hsv_hist.reshape(len(rgb), -1) / 1024.0)
        flat = rgb.reshape(len(rgb), -1, 3)
        mean = flat.mean(axis=1)
        std = flat.std(axis=1)
        skew = np.mean(((flat - mean[:, None, :]) / np.maximum(std[:, None, :], 1e-6)) ** 3, axis=1)
        outputs["rgb_stats"].append(np.concatenate((mean, std, skew), axis=1))
        batch_number = begin // batch_size + 1
        if verbose == 1 and (
            batch_number % progress_interval == 0
            or batch_number == total_batches
        ):
            processed = min(begin + batch_size, len(x))
            print(f"Descriptor: {processed}/{len(x)} gambar", flush=True)
    result = {key: np.concatenate(parts).astype(np.float32) for key, parts in outputs.items()}
    assert [result[k].shape[1] for k in ("hog", "lbp", "hsv", "rgb_stats")] == [324, 256, 48, 9]
    assert all(np.isfinite(value).all() for value in result.values())
    return result

## 5. Ekstraksi empat descriptor
Proses bertahap untuk mengendalikan penggunaan RAM.

In [5]:
start = time.perf_counter()
handcrafted = handcrafted_descriptors(x_all, batch_size=128, verbose=1)
print("Detik descriptor:", time.perf_counter() - start)
print({key: value.shape for key, value in handcrafted.items()})

Descriptor: 2432/50000 gambar
Descriptor: 4864/50000 gambar
Descriptor: 7296/50000 gambar
Descriptor: 9728/50000 gambar
Descriptor: 12160/50000 gambar
Descriptor: 14592/50000 gambar
Descriptor: 17024/50000 gambar
Descriptor: 19456/50000 gambar
Descriptor: 21888/50000 gambar
Descriptor: 24320/50000 gambar
Descriptor: 26752/50000 gambar
Descriptor: 29184/50000 gambar
Descriptor: 31616/50000 gambar
Descriptor: 34048/50000 gambar
Descriptor: 36480/50000 gambar
Descriptor: 38912/50000 gambar
Descriptor: 41344/50000 gambar
Descriptor: 43776/50000 gambar
Descriptor: 46208/50000 gambar
Descriptor: 48640/50000 gambar
Descriptor: 50000/50000 gambar
Detik descriptor: 17.34496491900063
{'hog': (50000, 324), 'lbp': (50000, 256), 'hsv': (50000, 48), 'rgb_stats': (50000, 9)}


## 6. Susun tujuh blok
Urutan blok tetap: RGB, AVG, NTSC, HOG, LBP, HSV, statistik RGB. Semua blok float32.

In [6]:
block_names = ("rgb", "avg", "ntsc", "hog", "lbp", "hsv", "rgb_stats")
blocks = [cnn_features[k] for k in ("rgb", "avg", "ntsc")]
blocks += [handcrafted[k] for k in ("hog", "lbp", "hsv", "rgb_stats")]
boundaries = np.cumsum([0] + [part.shape[1] for part in blocks]).astype(np.int32)
combined = np.concatenate(blocks, axis=1).astype(np.float32)
assert combined.shape == (50000, 2173) and np.isfinite(combined).all()
print("Batas blok:", boundaries, "Bentuk:", combined.shape)

Batas blok: [   0  512 1024 1536 1860 2116 2164 2173] Bentuk: (50000, 2173)


## 7. Simpan fitur gabungan
Sertakan metadata yang diperlukan Training dan Testing untuk memeriksa sumber fitur.

In [7]:
np.savez_compressed(OUT / "fitur" / "train_combined.npz", features=combined,
                    y=y_all, indices=np.arange(50000), train_idx=train_idx, val_idx=val_idx,
                    run_id=RUN_ID, split_hash=split_hash, preprocessing_version="v1",
                    block_names=np.array(block_names), boundaries=boundaries,
                    model_hashes=np.array([model_hashes[k] for k in ("rgb", "avg", "ntsc")]))
print(OUT / "fitur" / "train_combined.npz")

/home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v1/fusion/fitur/train_combined.npz
